<a href="https://www.kaggle.com/code/martinsertin/blueberry-yield-optuna-xgb?scriptVersionId=287575446" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
import optuna


In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])

In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    # ---- Bees ----
    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']

    # ---- Temperature ----
    data['temp_range'] = (
        data['MaxOfUpperTRange'] -
        data['MinOfLowerTRange']
    )

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']

    # ---- Non-linear ----
    for col in ['clonesize', 'total_bees', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    # ---- Clustering ----
    cluster_cols = ['clonesize', 'total_bees', 'avg_temp', 'RainingDays']

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])

        kmeans = KMeans(
            n_clusters=6,
            random_state=42,
            n_init=20
        )
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    return data, kmeans, scaler


In [4]:
train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

X = train_fe.drop(columns=['yield'])
y = np.log1p(train_fe['yield'])

y_min, y_max = train_fe['yield'].min(), train_fe['yield'].max()

In [5]:
def objective(trial):
    params = {
        'n_estimators': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
        'max_depth': trial.suggest_int('max_depth', 5, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 5),
        'objective': 'reg:absoluteerror',
        'tree_method': 'gpu_hist',
        'predictor': 'gpu_predictor',
        'random_state': 42
    }

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    maes = []

    for tr_idx, val_idx in kf.split(X):
        model = XGBRegressor(**params)

        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            early_stopping_rounds=200,
            verbose=False
        )

        preds = np.expm1(model.predict(X.iloc[val_idx]))
        true  = np.expm1(y.iloc[val_idx])

        maes.append(mean_absolute_error(true, preds))

    return np.mean(maes)

In [6]:
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=60)

best_params = study.best_params
print(f"Best CV MAE: {study.best_value:.4f}")
print("Best Params:", best_params)

[I 2025-12-21 07:28:45,706] A new study created in memory with name: no-name-605d3552-835f-4ae1-ac0c-a1d3deb928dc
[I 2025-12-21 07:29:13,932] Trial 0 finished with value: 246.363471305401 and parameters: {'learning_rate': 0.0249816047538945, 'max_depth': 8, 'min_child_weight': 8, 'subsample': 0.8496646210492591, 'colsample_bytree': 0.7390046601106091, 'gamma': 0.15599452033620265, 'reg_alpha': 0.05808361216819946, 'reg_lambda': 4.46470458309974}. Best is trial 0 with value: 246.363471305401.
[I 2025-12-21 07:29:33,787] Trial 1 finished with value: 245.85290225007807 and parameters: {'learning_rate': 0.034044600469728355, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.9424774630404985, 'colsample_bytree': 0.9081106602001054, 'gamma': 0.21233911067827616, 'reg_alpha': 0.18182496720710062, 'reg_lambda': 1.7336180394137353}. Best is trial 1 with value: 245.85290225007807.
[I 2025-12-21 07:29:59,824] Trial 2 finished with value: 245.33880887750516 and parameters: {'learning_rate': 0.

Best CV MAE: 244.6615
Best Params: {'learning_rate': 0.012891647030592175, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.7720152567327854, 'colsample_bytree': 0.846710169525111, 'gamma': 0.9188658201982547, 'reg_alpha': 0.6808074752852558, 'reg_lambda': 1.7942841615898426}


In [7]:
kf = KFold(n_splits=15, shuffle=True, random_state=42)

test_preds = np.zeros(len(test_fe))
oof_preds  = np.zeros(len(X))
maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"Fold {fold}/15")

    model = XGBRegressor(
        **best_params,
        n_estimators=5000,
        objective='reg:absoluteerror',
        tree_method='gpu_hist',
        predictor='gpu_predictor',
        random_state=42
    )

    model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
        early_stopping_rounds=200,
        verbose=False
    )

    val_pred = np.expm1(model.predict(X.iloc[val_idx]))
    true_val = np.expm1(y.iloc[val_idx])

    oof_preds[val_idx] = val_pred
    maes.append(mean_absolute_error(true_val, val_pred))

    test_preds += np.expm1(model.predict(test_fe)) / kf.n_splits

print(f"\nOOF MAE: {np.mean(maes):.4f} ± {np.std(maes):.4f}")

Fold 1/15
Fold 2/15
Fold 3/15
Fold 4/15
Fold 5/15
Fold 6/15
Fold 7/15
Fold 8/15
Fold 9/15
Fold 10/15
Fold 11/15
Fold 12/15
Fold 13/15
Fold 14/15
Fold 15/15

OOF MAE: 244.6831 ± 7.2683


In [8]:
test_preds = np.clip(test_preds, y_min, y_max)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7477.346619
1,15001,5900.817413
2,15002,6512.673187
3,15003,4629.406250
4,15004,5889.479187
